# Reusable Funnel / Multi-Event Analysis Template (R + dplyr)

Copy this notebook when you face a new set of related event tables (sign-ups → activation → conversion, applications → underwriting → funding, referrals → appointments → procedures, etc.).

**Replace the placeholders** marked with `YOUR_…` and follow the comments.

In [ ]:
library(readr)
library(dplyr)
library(lubridate)
library(ggplot2)
set.seed(42)

## 1. Load your tables

Earliest event = the grain you will left-join everything onto.

In [ ]:
# YOUR_PATHS
stage1 <- read_csv("data/YOUR_STAGE1.csv")   # e.g. visits / signups / applications
stage2 <- read_csv("data/YOUR_STAGE2.csv")   # e.g. checkouts / activations / underwritten
stage3 <- read_csv("data/YOUR_STAGE3.csv")   # e.g. purchases / paid / funded

# YOUR_KEY_COLUMN (must exist in all three, or rename first)
KEY <- "user_id"

# YOUR_TIME_COLUMNS
TIME1 <- "visit_time"
TIME2 <- "checkout_time"
TIME3 <- "purchase_time"

glimpse(stage1); glimpse(stage2); glimpse(stage3)
cat("Rows:", nrow(stage1), nrow(stage2), nrow(stage3), "\n")

## 2. Build the funnel table (left joins from stage 1)

In [ ]:
funnel <- stage1 %>%
  left_join(stage2, by = KEY) %>%
  left_join(stage3, by = KEY) %>%
  mutate(
    reached_stage2 = !is.na(.data[[TIME2]]),
    reached_stage3 = !is.na(.data[[TIME3]]),
    hrs_1_to_2 = as.numeric(difftime(.data[[TIME2]], .data[[TIME1]], units = "hours")),
    hrs_1_to_3 = as.numeric(difftime(.data[[TIME3]], .data[[TIME1]], units = "hours"))
  )

n1 <- nrow(funnel)
n2 <- sum(funnel$reached_stage2)
n3 <- sum(funnel$reached_stage3)

tibble(
  stage = c("Stage1", "Stage2", "Stage3"),
  count = c(n1, n2, n3),
  conv_from_prev = c(NA, n2/n1, n3/n2),
  conv_from_start = c(1, n2/n1, n3/n1)
)

## 3. Quick plots

In [ ]:
# Absolute counts
tibble(stage = factor(c("S1","S2","S3"), levels = c("S1","S2","S3")),
       count = c(n1, n2, n3)) %>%
  ggplot(aes(stage, count, fill = stage)) + geom_col() + theme_minimal() + theme(legend.position = "none")

# Time distribution (stage 1 → 2)
funnel %>% filter(reached_stage2) %>%
  ggplot(aes(hrs_1_to_2)) + geom_histogram(bins = 20, fill = "#16a085") + theme_minimal()

## 4. Optional simulation stub

Copy the more complete simulation cell from the Cool T-Shirts Solution notebook and adapt the probability / time parameters to your domain.

In [ ]:
# Placeholder — paste simulation code here when needed